In [1]:
"""
Bivariate polynomial addition
"""
from wikiversity import init_tables

gf_log, gf_exp = init_tables(prim=0x11b)

# If you wish to represent instead of the elements themselves the powers of the primitive element, 1 or 0
# (since the element depends on the primitive polynomial used this might be a good representation)
# it is simply done by taking the result and performing its discrete logarithm (gf_log[result])
# gf_log[0] = 0 (zero element)
# gf_log[1] = 0 (a^0, or the one element)

def gf_bipoly_add(x, y):
    """
    Adds two GF(2^p) bivariate polynomials represented as matrices.

    Args:
        x: The first bivariate polynomial (matrix).
        y: The second bivariate polynomial (matrix).

    Returns:
        The sum of the two bivariate polynomials (matrix).
    """
    return [[x_ij ^ y_ij for x_ij, y_ij in zip(row_x, row_y)]
            for row_x, row_y in zip(x, y)]

def gf_bipoly_sub(x, y):
    """
    Subtracts two GF(2^p) bivariate polynomials represented as matrices.

    Args:
        x: The first bivariate polynomial (matrix).
        y: The second bivariate polynomial (matrix).

    Returns:
        The subtraction of the two bivariate polynomials (matrix).
    """
    return gf_bipoly_add(x, y)

In [2]:
rows = 6    # max(x_degree)+1
cols = 7 # max(y_degree)+1

bip = [[0 for _ in range(cols)] for _ in range(rows)]

bip2 = [[0 for _ in range(cols)] for _ in range(rows)]

# setting bip = (a^41)x^5y^3+(a^20)x^3y^6+(a^12)x^3y^3+(a^14)
bip[0][0] = gf_exp[14]
bip[3][3] = gf_exp[12]
bip[3][6] = gf_exp[20]
bip[5][3] = gf_exp[41]
print(bip)
# setting bip2 = (a^46)x^3y^6
bip2[3][6] = gf_exp[46]

gf_bipoly_add(bip, bip2)

[[19, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 161, 0, 0, 216], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 190, 0, 0, 0]]


[[19, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 161, 0, 0, 62],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 190, 0, 0, 0]]

In [3]:
"""
Monomial multiplication
"""
from wikiversity import gf_mul
def gf_bipoly_monomul(bipoly, b_coefficient, b_x_deg, b_y_deg):
    """
    Multiplies a GF(2) bi-polynomial by a monomial.

    Args:
        bipoly: The bi-polynomial (matrix).
        b_coefficient: The coefficient of the monomial.
        b_x_deg: The degree of x in the monomial.
        b_y_deg: The degree of y in the monomial.

    Returns:
        A new bi-polynomial (matrix) representing the product.

    Raises:
        ValueError: If b_x_deg or b_y_deg is negative.
    """

    if b_x_deg < 0 or b_y_deg < 0:
        raise ValueError("Monomial powers must be non-negative")

    x_deg = len(bipoly) - 1  # Max x degree of the bi-polynomial
    y_deg = len(bipoly[0]) - 1  # Max y degree of the bi-polynomial

    res_x_deg = x_deg + b_x_deg
    res_y_deg = y_deg + b_y_deg

    # Initialize the result matrix with zeros
    resCoef = [[0 for _ in range(res_y_deg + 1)] for _ in range(res_x_deg + 1)]

    # Perform the multiplication
    for i in range(x_deg + 1):
        for j in range(y_deg + 1):
            resCoef[i + b_x_deg][j + b_y_deg] = gf_mul(bipoly[i][j], b_coefficient)

    return resCoef


In [4]:
gf_bipoly_monomul(bip, gf_exp[237], 3, 8)

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 180, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 36, 0, 0, 5],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 164, 0, 0, 0]]

In [5]:
"""
Binomial Coefficient
"""
def comb(m, n):
    """
    Calculates the binomial coefficient "m choose n" (number of ways to choose n items from a set of m items).

    Args:
        m: The total number of items.
        n: The number of items to choose.

    Returns:
        The binomial coefficient (m choose n).
    """

    n_ = min(n, m - n)  # Optimization: choose the smaller value
    if n_ < 0:
        return 0  # Invalid input
    if n_ == 0:
        return 1  # Base case: choosing 0 items
    if n_ == 1:
        return m  # Base case: choosing 1 item

    L = [0] * n_  # Initialize a list of n_ zeros

    for k in range(m):
        L_i_1 = 1
        for i in range(n_):
            L_i = L[i]
            L[i] = L[i] + L_i_1
            L_i_1 = L_i

    return L[n_ - 1]

In [6]:
print(comb(21, 17))
print(comb(12, 34))
print(comb(43, 21))

5985
0
1052049481860


In [7]:
from wikiversity import gf_add, init_tables, gf_mul, gf_pow

gf_log, gf_exp = init_tables(prim=0x11b)

def gf_bipoly_get_coeff(bipoly, i, j, gf_zero):
    """
    Gets the coefficient of x^i * y^j in the bi-polynomial, handling out-of-bounds indices.

    Args:
        bipoly: The bi-polynomial (matrix).
        i: The power of x.
        j: The power of y.
        gf_zero: The zero element of the Galois Field.

    Returns:
        The coefficient of x^i * y^j, or gf_zero if the index is out of bounds.
    """
    x_degree = len(bipoly) - 1
    y_degree = len(bipoly[0]) - 1

    if 0 <= i <= x_degree and 0 <= j <= y_degree:
        return bipoly[i][j]
    else:
        return gf_zero


def gf_bipoly_coeff_shifted(bipoly, x0, y0, i, j, gf_zero):
    """
    Calculates the coefficient of (x - x0)^i * (y - y0)^j in the bi-polynomial.

    Args:
        bipoly: The bi-polynomial (matrix).
        x0: The x-shift value.
        y0: The y-shift value.
        i: The power of (x - x0).
        j: The power of (y - y0).
        gf_zero: The zero element of the Galois Field.

    Returns:
        The coefficient of (x - x0)^i * (y - y0)^j.
    """

    x_degree = len(bipoly) - 1
    y_degree = len(bipoly[0]) - 1

    if x0 == gf_zero and y0 == gf_zero:
        return gf_bipoly_get_coeff(bipoly, i, j, gf_zero)
    elif x0 == gf_zero:
        res = gf_zero
        for jj in range(j, y_degree + 1):
            res = gf_add(res, gf_mul(comb(jj, j), gf_mul(gf_bipoly_get_coeff(bipoly, i, jj, gf_zero), gf_pow(y0, jj - j))))
        return res
    elif y0 == gf_zero:
        res = gf_zero
        for ii in range(i, x_degree + 1):
            res = gf_add(res, gf_mul(comb(ii, i), gf_mul(gf_bipoly_get_coeff(bipoly, ii, j, gf_zero), gf_pow(x0, ii - i))))
        return res
    else:
        res = gf_zero
        for ii in range(i, x_degree + 1):
            for jj in range(j, y_degree + 1):
                res = gf_add(res, gf_mul(comb(ii, i) * comb(jj, j), gf_mul(gf_bipoly_get_coeff(bipoly, ii, jj, gf_zero), gf_mul(gf_pow(x0, ii) - i, gf_pow(y0, jj - j)))))
        return res


In [8]:
from wikiversity import init_tables
gf_log, gf_exp = init_tables(prim=0x11b)
print(gf_exp[8])
print(gf_exp[17])
gf_bipoly_coeff_shifted(bip, gf_exp[8], gf_exp[17], 3, 2, 0)

26
225


129

In [9]:
"""
Bivariate polynomial multiplication
"""
from wikiversity import gf_mul